In [1]:
import os
import pandas as pd
import kagglehub
from IPython.display import display
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
import fitz

# 1) Data preparation 

In [2]:
# Excel file
df_excel = pd.read_excel("/Users/constancehenin/Documents/JOB RECHERCHE/Recherche CDI - 2025/GITHUB PORTFOLIO/Project_1/preprocessed-50k.xlsx")

print(df_excel.sample(5))

# Format the dataframe
df = pd.DataFrame(df_excel)

# Excel
df.info()
df["generated"].value_counts()

# Sample the whole dataset to run faster
df_sample = df.sample(n=20000, random_state=123).reset_index(drop=True)

df_sample.info()
print(df_sample.sample(n=10))
df_sample["generated"].value_counts()

                                                    text  generated
3990   Description\n\nLong ago, in the far reaches of...          0
37477  I'm sorry, but it's not appropriate to discuss...          1
4168   I can control minds. \n \n I need eye contact ...          0
44243   The best way to keep your health costs down i...          1
28374  Cristiano Ronaldo's move to Real Madrid from M...          1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       50000 non-null  object
 1   generated  50000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 781.4+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       20000 non-null  object
 1   generated  20000 non-null  int64 
dtypes: int6

generated
1    10138
0     9862
Name: count, dtype: int64

# 2) Model planning

In [3]:
# Fix the target and the features for modeling

X_text = df_sample["text"]
X_text = [str(x) if x is not None else "" for x in X_text]
X_text_series = pd.Series(X_text)
y = df_sample["generated"]

## Data (text) vectorization

vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    lowercase=True,
    ngram_range=(1, 3),
    min_df=5,
    token_pattern=r"(?u)\b\w+\b|[^\w\s]"  # keeps words + ponctuation
)

X = vectorizer.fit_transform(X_text)

# Split df into two sets (training & testing)
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=123, stratify=y
)

# 3) Model building

In [4]:
## Logistic Regression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Random Forest

rfc = RandomForestClassifier(n_estimators=100, random_state=42)
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## XGBoost

xgb_ = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_.fit(X_train, y_train)
y_pred = xgb_.predict(X_test)

print("=== XGBoost ===")
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.title("Confusion Matrix - XGBoost")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## LightGBM

lgb_ = lgb.LGBMClassifier()
lgb_.fit(X_train, y_train)
y_pred = lgb_.predict(X_test)

print("=== LightGBM ===")
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.title("Confusion Matrix - LightGBM")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Gradient Boosting

gbc = GradientBoostingClassifier()
gbc.fit(X_train, y_train)
y_pred = gbc.predict(X_test)

print("=== Gradient Boosting ===")
print(classification_report(y_test, y_pred, target_names=["Human", "AI"]))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Human", "AI"], yticklabels=["Human", "AI"])
plt.title("Confusion Matrix - Gradient Boosting")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

=== Logistic Regression ===
              precision    recall  f1-score   support

       Human       0.88      0.78      0.82      1972
          AI       0.80      0.89      0.85      2028

    accuracy                           0.84      4000
   macro avg       0.84      0.84      0.84      4000
weighted avg       0.84      0.84      0.84      4000



/var/folders/bb/02frpy2907d89x9qmmzbq3rc0000gn/T/ipykernel_3203/3498695737.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== Random Forest ===
              precision    recall  f1-score   support

       Human       0.88      0.77      0.82      1972
          AI       0.80      0.89      0.84      2028

    accuracy                           0.83      4000
   macro avg       0.84      0.83      0.83      4000
weighted avg       0.84      0.83      0.83      4000



/var/folders/bb/02frpy2907d89x9qmmzbq3rc0000gn/T/ipykernel_3203/3498695737.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:51:29] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost ===
              precision    recall  f1-score   support

       Human       0.86      0.80      0.83      1972
          AI       0.81      0.87      0.84      2028

    accuracy                           0.83      4000
   macro avg       0.84      0.83      0.83      4000
weighted avg       0.83      0.83      0.83      4000



/var/folders/bb/02frpy2907d89x9qmmzbq3rc0000gn/T/ipykernel_3203/3498695737.py:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[LightGBM] [Info] Number of positive: 8110, number of negative: 7890
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.340528 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 484606
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 16627
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.506875 -> initscore=0.027502
[LightGBM] [Info] Start training from score 0.027502
=== LightGBM ===
              precision    recall  f1-score   support

       Human       0.87      0.80      0.83      1972
          AI       0.82      0.89      0.85      2028

    accuracy                           0.84      4000
   macro avg       0.85      0.84      0.84      4000
weighted avg       0.84      0.84      0.84      4000



/var/folders/bb/02frpy2907d89x9qmmzbq3rc0000gn/T/ipykernel_3203/3498695737.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


=== Gradient Boosting ===
              precision    recall  f1-score   support

       Human       0.87      0.71      0.78      1972
          AI       0.76      0.90      0.82      2028

    accuracy                           0.80      4000
   macro avg       0.81      0.80      0.80      4000
weighted avg       0.81      0.80      0.80      4000



/var/folders/bb/02frpy2907d89x9qmmzbq3rc0000gn/T/ipykernel_3203/3498695737.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# 4) Model Evaluation

In [5]:
# Dictionnary to store the performances
performance_summary = {}

# Function to extract the metrics
def evaluate_model(name, y_test, y_pred):
    report = classification_report(y_test, y_pred, output_dict=True)
    performance_summary[name] = {
        "Accuracy": report["accuracy"],
        "Precision (AI)": report["1"]["precision"],
        "Recall (AI)": report["1"]["recall"],
        "F1-score (AI)": report["1"]["f1-score"],
        "Precision (Human)": report["0"]["precision"],
        "Recall (Human)": report["0"]["recall"],
        "F1-score (Human)": report["0"]["f1-score"]
    }

# Evaluate the models

# Logistic Regression
evaluate_model("Logistic Regression", y_test, LogisticRegression().fit(X_train, y_train).predict(X_test))

# Random Forest
evaluate_model("Random Forest", y_test, RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train).predict(X_test))

# XGBoost
evaluate_model("XGBoost", y_test, xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss').fit(X_train, y_train).predict(X_test))

# LightGBM
evaluate_model("LightGBM", y_test, lgb.LGBMClassifier().fit(X_train, y_train).predict(X_test))

# Gradient Boosting
evaluate_model("Gradient Boosting", y_test, GradientBoostingClassifier().fit(X_train, y_train).predict(X_test))

# Print the performances
summary_df = pd.DataFrame(performance_summary).T.round(3)
print("=== Comparative summary of performances ===")
display(summary_df.sort_values(by="Precision (AI)", ascending=False))

/opt/anaconda3/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [15:52:57] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[LightGBM] [Info] Number of positive: 8110, number of negative: 7890
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.302026 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 484606
[LightGBM] [Info] Number of data points in the train set: 16000, number of used features: 16627
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.506875 -> initscore=0.027502
[LightGBM] [Info] Start training from score 0.027502
=== Comparative summary of performances ===


,Accuracy,Precision (AI),Recall (AI),F1-score (AI),Precision (Human),Recall (Human),F1-score (Human)
LightGBM,0.842,0.818,0.886,0.851,0.872,0.798,0.833
XGBoost,0.834,0.815,0.869,0.841,0.856,0.797,0.825
Logistic Regression,0.836,0.805,0.895,0.847,0.878,0.776,0.824
Random Forest,0.833,0.801,0.893,0.844,0.875,0.771,0.820
Gradient Boosting,0.804,0.760,0.896,0.822,0.869,0.708,0.781


# After we have choose the best model we will use it to develop it in a app online using stremlit 